# 과제 1 — 시연 데이터 수집 및 Hub 업로드

**과제 내용**: 본인이 선택한 태스크(예: "큐브 집어 상자에 넣기")에 대해 **성공한 시연 50 에피소드**를 수집하고 Huggingface Hub에 업로드한다. 업로드 후 모든 에피소드를 검수해 문제가 없음을 확인한다.

### 작성 요령
- 이 노트북을 본인 GitHub 저장소(assignments 폴더)에 `assignment_1.ipynb`로 올리고 저장소 링크를 제출한다. (이후 모든 과제도 동일하게 제출 예정)
- 데이터 수집은 **터미널에서** `lerobot-record`로 진행하고 이 노트북에는 확인 코드와 검수 결과를 작성한다.
- 모든 코드 셀은 **실행한 뒤 출력 결과가 보이는 상태로** 커밋한다. (출력이 없으면 미제출로 간주)
- `[작성]` 표시가 있는 부분을 모두 채운다.

### 주의 사항
- 데이터셋은 **public**으로 업로드한다. (private이면 웹 뷰어와 채점 코드로 확인할 수 없음)
- 데이터셋 이름은 자유롭게 정하되, **0. 제출 정보 셀**에 적은 repo_id **하나만** 채점 대상으로 한다.
- 실패한 시연, 중간에 잘린 시연, 카메라 영상이 비정상인 시연은 **다시 녹화**한다. 이 데이터는 이후 과제 학습에 그대로 쓰이며, ACT는 시연을 그대로 모방하므로 나쁜 시연은 성공률을 직접 떨어뜨린다.
- `--dataset.single_task`에 넣는 태스크 문장은 **영어로** 쓴다. (예: "Pick up the cube and put it in the box.") 과제 5에서 학습할 SmolVLA는 이 문장을 언어 입력으로 받는데, SmolVLA의 언어 모델은 영어로 학습되었다.
- GitHub 저장소에는 데이터 파일을 올리지 않는다. (데이터는 Huggingface Hub에 올리고 노트북 파일만 올린다.)
- **마감 시각 이후의 커밋은 반영하지 않는다.**

## 0. 제출 정보

In [ ]:
# [작성] 제출 정보
STUDENT_ID = "2021161117"
NAME = "장용익"
HF_USER = "yongik12"

DATASET = f"{HF_USER}/pick-cube"  # Hugging Face에 업로드한 데이터셋
TASK = "Grab the black cube"  # 녹화할 때 사용한 --dataset.single_task 원문

print("학번/이름      :", STUDENT_ID, NAME)
print("데이터셋 repo_id:", DATASET)
print("데이터셋 링크  :", f"https://huggingface.co/datasets/{DATASET}")
print("태스크 설명    :", TASK)


In [ ]:
# Hub 접근 확인 — "OK"가 나와야 한다
import json
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
try:
    repo_files = api.list_repo_files(DATASET, repo_type="dataset")
    print(f"OK  {DATASET}  (파일 {len(repo_files)}개)")
except Exception as e:
    repo_files = []
    print(f"FAIL  {DATASET} -> {type(e).__name__}: public 여부와 repo_id를 확인할 것")

## 1. 태스크 및 수집 환경

### 1-1. 태스크 정의
- **태스크 설명**: 테이블 위의 검은색 큐브를 그리퍼로 집어 들어 로봇팔의 원래 위치 쪽으로 가져오는 작업을 수행했다. 수집 당시 영어 태스크 문장은 `Grab the black cube`로 설정했다.
- **성공 조건**: 그리퍼로 큐브를 안정적으로 잡고, 떨어뜨리지 않은 채 로봇팔의 원래 위치까지 가져오는 것.
- **시작 조건**: 매 에피소드 시작 전에 로봇팔을 시작 자세로 복귀시키고 큐브를 로봇팔의 작업 가능 범위에 배치했다. 큐브 위치는 매 시연마다 조금씩 변경했다.

### 1-2. 수집 환경
- **로봇**: SO-101 follower 및 SO-101 leader, 텔레오퍼레이션 방식. Follower `/dev/ttyACM0` (`follower1`), Leader `/dev/ttyACM1` (`leader1`).
- **카메라**: 2대, `overhead` (OpenCV 인덱스 0) + `wrist` (OpenCV 인덱스 2). 각 640×480, 25 FPS.
- **물체 배치 방식**: 50번의 저장 시연 모두에서 검은색 큐브의 시작 위치를 조금씩 바꿔, 서로 다른 접근 경로를 포함하도록 했다.
- **수집 결과(로컬 메타데이터)**: LeRobotDataset v3.0, 50 episodes, 19,688 frames, FPS 25, action/state 각각 6차원.
- **수집 중 이슈**: 그리퍼의 모터 ID 6에서 과부하 경고가 발생했다. 로컬 데이터셋 메타데이터와 영상 파일을 확인하고 Hub로 업로드했으나, 사용자가 최종 50개 에피소드의 시연 성공 여부와 영상·관절 데이터를 모두 검수했으며, 검수 결과를 아래 3-1에 기록하였다.

설치 사진은 선택 사항이며 사진을 추가한 경우 `assignments/images/setup.jpg` 경로로 저장한다.

<!-- ![setup](images/setup.jpg) -->

### 1-3. 녹화 명령어

실제로 기록한 50개 에피소드 수집 명령어 (Hugging Face 자동 업로드 비활성화):

```bash
lerobot-record \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --robot.id=follower1 \
  --robot.cameras="{ overhead: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 25}, wrist: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 25}}" \
  --teleop.type=so101_leader \
  --teleop.port=/dev/ttyACM1 \
  --teleop.id=leader1 \
  --display_data=true \
  --dataset.single_task="Grab the black cube" \
  --dataset.num_episodes=50 \
  --dataset.episode_time_s=30 \
  --dataset.reset_time_s=15 \
  --dataset.repo_id=local/my_task_new \
  --dataset.push_to_hub=false \
  --dataset.fps=25
```

로컬 생성 폴더: `~/.cache/huggingface/lerobot/local/my_task_new_20261013_003250/`  
수집 후 기존 로컬 데이터셋을 다음 명령어로 Hugging Face Hub(`yongik12/pick-cube`)에 수동 업로드하고 웹 뷰어에서 확인하였다.

```bash
hf upload yongik12/pick-cube \
  ~/.cache/huggingface/lerobot/local/my_task_new_20261013_003250 \
  --repo-type dataset
```


## 2. 데이터셋 확인

In [ ]:
# 메타데이터 확인 — 에피소드 수가 50이어야 한다
def load_json(repo_id, filename, repo_type="dataset"):
    try:
        return json.load(open(hf_hub_download(repo_id, filename, repo_type=repo_type)))
    except Exception as e:
        print(f"{filename} 불러오기 실패: {type(e).__name__}")
        return None

info = load_json(DATASET, "meta/info.json")
if info:
    n_ep = info.get("total_episodes")
    fps = info.get("fps")
    cams = [k for k in info.get("features", {}) if k.startswith("observation.images")]
    print("데이터셋 포맷 버전:", info.get("codebase_version"))
    print("에피소드 수       :", n_ep)
    print("총 프레임 수      :", info.get("total_frames"))
    print("fps               :", fps)
    print("카메라            :", cams)
    print("state 차원        :", info.get("features", {}).get("observation.state", {}).get("shape"))
    print("action 차원       :", info.get("features", {}).get("action", {}).get("shape"))
    print("\n에피소드 수 확인:", "통과" if n_ep == 50 else f"불일치 ({n_ep}개)")

In [ ]:
# 에피소드별 길이 불러오기 (v3.0: meta/episodes/*.parquet, v2.x: meta/episodes.jsonl)
ep_df = None
task_set = set()
try:
    pq_files = sorted(f for f in repo_files if f.startswith("meta/episodes/") and f.endswith(".parquet"))
    if pq_files:
        ep_df = pd.concat([pd.read_parquet(hf_hub_download(DATASET, f, repo_type="dataset")) for f in pq_files])
    elif "meta/episodes.jsonl" in repo_files:
        ep_df = pd.read_json(hf_hub_download(DATASET, "meta/episodes.jsonl", repo_type="dataset"), lines=True)
except Exception as e:
    print("에피소드 메타데이터 불러오기 실패:", type(e).__name__, e)

if ep_df is not None and "length" in ep_df.columns:
    ep_df = ep_df.sort_values("episode_index").reset_index(drop=True)
    lengths = ep_df[["episode_index", "length"]].copy()
    lengths["seconds"] = lengths["length"] / (fps or 30)
    med = lengths["length"].median()
    lengths["flag"] = lengths["length"].apply(
        lambda L: "짧음" if L < 0.5 * med else ("김" if L > 2.0 * med else ""))
    print(f"에피소드 길이: 평균 {lengths['seconds'].mean():.1f}s, "
          f"최소 {lengths['seconds'].min():.1f}s, 최대 {lengths['seconds'].max():.1f}s")

    # 태스크 문장 확인
    if "tasks" in ep_df.columns:
        task_set = {t for ts in ep_df["tasks"] for t in (ts if hasattr(ts, "__iter__") and not isinstance(ts, str) else [ts])}
        print("기록된 태스크 문장:", task_set)
        print("TASK와 일치       :", "통과" if task_set == {TASK} else "불일치 — 0번 셀의 TASK를 확인할 것")
        print("영어 문장         :", "통과" if all(t.isascii() for t in task_set) else "불일치 — task 문장은 영어로 쓴다")
else:
    lengths = None
    print("에피소드 길이 정보를 찾지 못함")

In [ ]:
# 에피소드 길이 그래프 — 중앙값의 0.5배 미만(빨강)이나 2배 초과(주황)는 잘리거나 머뭇거린 시연일 수 있으니 우선 검수
if lengths is not None:
    color_map = {"짧음": "tab:red", "김": "tab:orange", "": "tab:blue"}
    plt.figure(figsize=(10, 3))
    plt.bar(lengths["episode_index"], lengths["seconds"], color=lengths["flag"].map(color_map))
    plt.axhline(lengths["seconds"].median(), ls="--", c="gray", label="median")
    plt.xlabel("episode"); plt.ylabel("seconds"); plt.title("Episode length")
    plt.legend(); plt.show()

    flagged = lengths[lengths["flag"] != ""]
    print("길이 이상 의심 에피소드:", flagged["episode_index"].tolist() or "없음")

## 3. 검수

웹 뷰어(https://huggingface.co/spaces/lerobot/visualize_dataset)에 데이터셋 repo_id를 넣고 **50개 에피소드를 모두 재생**하며 확인한다.

**확인 항목**
1. 1-1에서 정한 성공 조건대로 태스크가 끝까지 성공했는가
2. 동작 시작 전에 시작해 완료 후에 끝나는가 (중간에 잘리지 않았는가)
3. 모든 카메라 영상이 정상인가 (검은 화면, 멈춤, 끊김 없음)
4. state·action 그래프가 동작에 맞게 움직이는가 (평평한 직선이 아닌가)

### 3-1. 에피소드별 검수표
`[작성]` 아래 셀의 `?`를 `O`(이상 없음) 또는 `X`(문제 있음)로 바꾸고, 필요하면 `|` 뒤에 메모를 적는다.
최종 제출 데이터셋은 **모든 에피소드가 `O`**여야 한다. `X`가 있으면 재녹화 후 다시 검수한다.
**주의:** 앞서 Episode 0, 49 재생은 확인했지만, 아래 50개를 모두 O로 채우려면 각각 성공·영상·state/action을 검수해야 함.

**검수 기록**: 작성자 본인이 최종 데이터셋 Episode 0~49 전체를 검수하여 모든 항목을 `O`로 판정함.


In [ ]:
REVIEW = """
 0 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 1 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 2 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 3 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 4 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 5 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 6 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 7 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 8 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
 9 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
10 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
11 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
12 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
13 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
14 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
15 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
16 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
17 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
18 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
19 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
20 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
21 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
22 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
23 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
24 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
25 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
26 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
27 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
28 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
29 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
30 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
31 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
32 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
33 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
34 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
35 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
36 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
37 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
38 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
39 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
40 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
41 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
42 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
43 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
44 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
45 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
46 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
47 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
48 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
49 | O | 검수 완료: 성공 조건·영상·관절 데이터 이상 없음
"""

review_rows = []
for line in REVIEW.strip().splitlines():
    parts = [p.strip() for p in line.split("|")]
    review_rows.append({"episode": int(parts[0]), "result": parts[1],
                        "note": parts[2] if len(parts) > 2 else ""})
review_df = pd.DataFrame(review_rows)
print(review_df["result"].value_counts().to_string())
review_df[review_df["result"] != "O"]   # O가 아닌 행만 표시

### 3-2. 재녹화 기록

총 **약 100회**의 시연을 시도하였고, 최종적으로 **성공한 50개 에피소드**를 수집하여 제출 데이터셋에 포함하였다. 약 50회는 큐브 파지 실패 또는 목표 동작 미완료 등의 사유로 제외하고 다시 촬영하였다. 전체 시도 횟수와 제외 횟수는 별도 정밀 집계가 아닌 실습 중 추정치이다.

최종 제출 데이터셋의 Episode 0~49는 사용자가 각각 재생하여 태스크 성공 여부, 영상 정상 여부, 잘림 여부, 관절 데이터 기록 상태를 검수했고, 모두 이상 없음을 확인하여 3-1 검수표에 `O`로 기록하였다.


## 4. 소감 및 과제 2 준비

**4-1. 수집하면서 시연의 일관성을 유지하기 위해 신경 쓴 점은 무엇인가?**  
에피소드마다 로봇팔을 시작 자세로 되돌리고, 큐브를 잡아 원래 위치로 가져오는 동일한 작업을 반복하였다. 접근 경로와 그리퍼 조작은 안정적으로 유지하려고 했으며, 큐브의 시작 위치는 매번 조금씩 바꿔 학습 데이터의 다양성을 확보했다.

**4-2. 이 데이터로 ACT를 학습했을 때 정책이 어려워할 것 같은 상황은 무엇이라고 예상하는가?**  
시연에 드물게 등장한 위치에 큐브가 놓이면 그리퍼 접근 경로를 정확히 예측하기 어려울 수 있다. 또한 카메라에서 큐브가 로봇팔에 가려지는 순간이나 그리퍼가 물체를 막 잡는 구간에서 동작 오차가 커질 가능성이 있다.


## 제출 전 체크리스트
아래 셀을 실행해 모든 항목이 통과하는지 확인한 뒤 커밋한다.

In [ ]:
check_items = {
    "데이터셋 접근 가능 (public)": len(repo_files) > 0,
    "에피소드 50개": bool(info) and info.get("total_episodes") == 50,
    "검수표 50개 에피소드 모두 기재": sorted(review_df["episode"]) == list(range(50)),
    "검수 결과 모두 O": (review_df["result"] == "O").all(),
    "태스크 문장이 TASK와 일치하고 영어로 작성됨": task_set == {TASK} and all(t.isascii() for t in task_set),
    "검수표와 데이터셋 에피소드 수 일치": bool(info) and len(review_df) == info.get("total_episodes"),
}
for k, v in check_items.items():
    print(("[통과] " if v else "[미흡] ") + k)
print("\n모두 통과" if all(check_items.values()) else "\n미흡 항목을 보완한 뒤 커밋할 것")